# Primer Parcial de Ciencias de Datos

Este notebook documenta el desarrollo de un modelo predictivo para estimar la probabilidad de adquisición de un producto financiero. El trabajo incluye una validación por cliente, comparación entre una línea base y una versión mejorada, y la generación del archivo final de predicciones.

Archivo de salida:
- `bank_product_propensity_predictions.csv`

## Metodología

1. Se realiza una exploración inicial del conjunto de datos y de la distribución de la variable objetivo.
2. Se construye una partición de entrenamiento y validación agrupada por `customer_id` para evitar fuga de información entre observaciones del mismo cliente.
3. Se establece una línea base con variables originales para contar con un punto de referencia.
4. Se incorpora ingeniería de variables, selección embebida de características y reentrenamiento del modelo final.
5. Se ajusta el modelo definitivo sobre todo el conjunto de entrenamiento y se generan las predicciones requeridas para la entrega.

Se agrega el siguiente código para poder verificar con cual Kerner se está ejecutando y validar que la instalación de todos los componentes seá correcta:

In [ ]:
import sys
print(sys.executable)

Se agrega workarround para la instalación de paquetes por si falla la instalación por consola:

In [ ]:
import sys
!"{sys.executable}" -m pip install numpy pandas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "bank_product_propensity_dataset.csv"
PREDICT_PATH = ROOT / "bank_product_propensity_predict.csv"
OUTPUT_PATH = ROOT / "bank_product_propensity_predictions.csv"

TARGET_COL = "target_product_acquired"
GROUP_COL = "customer_id"
DATE_COL = "snapshot_date"
CAT_COLS = ["employment_status", "education_level", "marital_status"]
TEMPORAL_COLS = ["snapshot_year", "snapshot_month", "snapshot_quarter", "snapshot_mindex"]
ENGINEERED_COLS = [
    "total_relationship_products",
    "raw_relationship_flags",
    "wealth_total_balance",
    "net_cashflow_last_month",
    "cashflow_ratio_last_month",
    "digital_engagement_score",
    "service_burden",
    "balance_total",
    "balance_to_income",
    "debt_burden",
    "wealth_index",
    "rfm_score_total",
    "risk_adjusted_value",
    "income_per_dependent",
    "products_per_tenure",
    "support_usage_rate",
    "trans_amount_per_txn",
    "deposit_minus_fee",
    "credit_buffer",
    "engagement_to_risk",
    "wealth_to_risk",
    "transactions_per_login",
    "investment_balance_share",
    "customer_value_per_product",
    "monthly_activity_score",
    "log1p_income",
    "log1p_savings_balance",
    "log1p_checking_balance",
    "log1p_credit_card_balance",
    "log1p_investment_balance",
    "log1p_total_transaction_amount_last_month",
    "log1p_total_deposit_amount_last_month",
    "log1p_total_withdrawal_amount_last_month",
    "log1p_customer_lifetime_value",
]


def sigmoid(values):
    clipped = np.clip(values, -35, 35)
    return 1.0 / (1.0 + np.exp(-clipped))


def unique_preserve_order(values):
    seen = set()
    ordered = []
    for value in values:
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered


def get_base_numeric_cols(df):
    excluded = {TARGET_COL, GROUP_COL, DATE_COL, *CAT_COLS}
    return [column for column in df.columns if column not in excluded]


def add_feature_engineering(df):
    data = df.copy()
    data[DATE_COL] = pd.to_datetime(data[DATE_COL], errors="coerce")
    data["snapshot_year"] = data[DATE_COL].dt.year.fillna(0).astype(float)
    data["snapshot_month"] = data[DATE_COL].dt.month.fillna(0).astype(float)
    data["snapshot_quarter"] = data[DATE_COL].dt.quarter.fillna(0).astype(float)
    data["snapshot_mindex"] = data["snapshot_year"] * 12.0 + data["snapshot_month"]

    def num(column):
        return pd.to_numeric(data[column], errors="coerce")

    def fill0(column):
        return num(column).fillna(0.0)

    relationship_flags = (
        fill0("has_savings_account")
        + fill0("has_checking_account")
        + fill0("has_credit_card")
        + fill0("has_mortgage")
        + fill0("has_personal_loan")
        + fill0("has_investment_account")
    )
    balance_total = (
        fill0("savings_balance")
        + fill0("checking_balance")
        + fill0("credit_card_balance")
        + fill0("investment_balance")
    )
    login_intensity = fill0("num_mobile_logins_last_month") + fill0("num_web_logins_last_month")

    data["raw_relationship_flags"] = relationship_flags
    data["total_relationship_products"] = fill0("num_products") + relationship_flags
    data["wealth_total_balance"] = balance_total
    data["net_cashflow_last_month"] = fill0("total_deposit_amount_last_month") - fill0("total_withdrawal_amount_last_month")
    data["cashflow_ratio_last_month"] = fill0("total_deposit_amount_last_month") / (fill0("total_withdrawal_amount_last_month") + 1.0)
    data["digital_engagement_score"] = (
        login_intensity
        + 10.0 * fill0("email_open_rate_last_3_months")
        + 25.0 * fill0("email_click_rate_last_3_months")
        + 20.0 * fill0("sms_response_rate_last_3_months")
    )
    data["service_burden"] = (
        fill0("num_customer_service_calls_last_month")
        + fill0("num_branch_visits_last_month")
        + fill0("num_failed_transactions_last_month")
    )
    data["balance_total"] = balance_total
    data["balance_to_income"] = balance_total / (fill0("income") + 1.0)
    data["debt_burden"] = fill0("debt_to_income_ratio") + fill0("credit_utilization")
    data["wealth_index"] = balance_total + 0.1 * fill0("customer_lifetime_value")
    data["rfm_score_total"] = fill0("recency_score") + fill0("frequency_score") + fill0("monetary_score")
    data["risk_adjusted_value"] = fill0("customer_lifetime_value") / (fill0("risk_score") + 1.0)
    data["income_per_dependent"] = fill0("income") / (fill0("num_dependents") + 1.0)
    data["products_per_tenure"] = fill0("num_products") / (fill0("tenure_months") + 1.0)
    data["support_usage_rate"] = fill0("num_customer_service_calls_last_month") / (fill0("tenure_months") + 1.0)
    data["trans_amount_per_txn"] = fill0("total_transaction_amount_last_month") / (fill0("num_transactions_last_month") + 1.0)
    data["deposit_minus_fee"] = fill0("total_deposit_amount_last_month") - fill0("total_overdraft_fees_last_year")
    data["credit_buffer"] = 1.0 - fill0("credit_utilization")
    data["engagement_to_risk"] = data["digital_engagement_score"] / (fill0("risk_score") + 1.0)
    data["wealth_to_risk"] = balance_total / (fill0("risk_score") + 1.0)
    data["transactions_per_login"] = fill0("num_transactions_last_month") / (login_intensity + 1.0)
    data["investment_balance_share"] = fill0("investment_balance") / (balance_total + 1.0)
    data["customer_value_per_product"] = fill0("customer_lifetime_value") / (fill0("num_products") + 1.0)
    data["monthly_activity_score"] = (
        fill0("num_transactions_last_month")
        + fill0("num_deposits_last_month")
        + fill0("num_withdrawals_last_month")
        + fill0("num_bill_payments_last_month")
    )
    for column in [
        "income",
        "savings_balance",
        "checking_balance",
        "credit_card_balance",
        "investment_balance",
        "total_transaction_amount_last_month",
        "total_deposit_amount_last_month",
        "total_withdrawal_amount_last_month",
        "customer_lifetime_value",
    ]:
        data[f"log1p_{column}"] = np.log1p(fill0(column).clip(lower=0.0))
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    return data


def build_feature_table(df):
    data = df.copy()
    base_numeric_cols = get_base_numeric_cols(data)
    data["raw_missing_count"] = data[base_numeric_cols].isna().sum(axis=1).astype(float)
    for column in base_numeric_cols:
        data[f"missing_{column}"] = data[column].isna().astype(float)
    data = add_feature_engineering(data)
    return data, base_numeric_cols


def selected_numeric_features(base_numeric_cols, feature_mode):
    baseline = unique_preserve_order(
        base_numeric_cols
        + TEMPORAL_COLS
        + ["raw_missing_count"]
        + [f"missing_{column}" for column in base_numeric_cols]
    )
    if feature_mode == "baseline":
        return baseline
    return unique_preserve_order(baseline + ENGINEERED_COLS)


def prepare_matrices(df, feature_mode, state=None):
    table, base_numeric_cols = build_feature_table(df)
    numeric_columns = selected_numeric_features(base_numeric_cols, feature_mode)
    numeric_frame = table[numeric_columns].apply(pd.to_numeric, errors="coerce")
    categorical_frame = table[CAT_COLS].fillna("Missing").astype(str).apply(lambda series: series.str.strip())

    if state is None:
        medians = numeric_frame.median()
        numeric_filled = numeric_frame.fillna(medians)
        means = numeric_filled.mean()
        stds = numeric_filled.std(ddof=0).replace(0.0, 1.0)
        dummy_frame = pd.get_dummies(categorical_frame, drop_first=True)
        dummy_columns = dummy_frame.columns.tolist()
        state = {
            "feature_mode": feature_mode,
            "base_numeric_cols": base_numeric_cols,
            "numeric_columns": numeric_columns,
            "medians": medians,
            "means": means,
            "stds": stds,
            "dummy_columns": dummy_columns,
        }
    else:
        numeric_filled = numeric_frame.reindex(columns=state["numeric_columns"]).fillna(state["medians"]).fillna(0.0)
        means = state["means"]
        stds = state["stds"]
        dummy_frame = pd.get_dummies(categorical_frame, drop_first=True).reindex(columns=state["dummy_columns"], fill_value=0.0)

    numeric_scaled = (numeric_filled.reindex(columns=state["numeric_columns"]) - means) / stds
    feature_frame = pd.concat([numeric_scaled, dummy_frame], axis=1)
    feature_frame = feature_frame.reindex(sorted(feature_frame.columns), axis=1)
    feature_names = feature_frame.columns.tolist()
    X = feature_frame.to_numpy(dtype=float)
    y = None if TARGET_COL not in df.columns else df[TARGET_COL].astype(int).to_numpy()
    groups = df[GROUP_COL].to_numpy()
    return X, feature_names, y, groups, state


def stratified_group_split(df, test_size=0.2, random_state=42):
    group_target = df.groupby(GROUP_COL)[TARGET_COL].max()
    positive_groups = group_target[group_target == 1].index.to_numpy(copy=True)
    negative_groups = group_target[group_target == 0].index.to_numpy(copy=True)
    rng = np.random.default_rng(random_state)
    rng.shuffle(positive_groups)
    rng.shuffle(negative_groups)
    positive_test = max(1, int(round(len(positive_groups) * test_size)))
    negative_test = max(1, int(round(len(negative_groups) * test_size)))
    test_groups = set(positive_groups[:positive_test]).union(set(negative_groups[:negative_test]))
    test_mask = df[GROUP_COL].isin(test_groups)
    train_df = df.loc[~test_mask].copy().reset_index(drop=True)
    val_df = df.loc[test_mask].copy().reset_index(drop=True)
    return train_df, val_df


def roc_auc_manual(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    ranks = pd.Series(y_score).rank(method="average").to_numpy()
    positives = y_true == 1
    n_pos = int(positives.sum())
    n_neg = int(len(y_true) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    rank_sum = ranks[positives].sum()
    return float((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def average_precision_manual(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    cumulative_tp = np.cumsum(y_sorted)
    precision = cumulative_tp / (np.arange(len(y_sorted)) + 1)
    positives = int(y_sorted.sum())
    if positives == 0:
        return float("nan")
    return float(precision[y_sorted == 1].sum() / positives)


def precision_recall_f1_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def best_threshold_for_f1(y_true, y_score):
    thresholds = np.unique(np.quantile(np.asarray(y_score, dtype=float), np.linspace(0.02, 0.98, 99)))
    best_threshold = 0.5
    best_f1 = -1.0
    best_precision = 0.0
    best_recall = 0.0
    for threshold in thresholds:
        prediction = (y_score >= threshold).astype(int)
        precision, recall, f1 = precision_recall_f1_manual(y_true, prediction)
        if f1 > best_f1:
            best_threshold = float(threshold)
            best_f1 = float(f1)
            best_precision = float(precision)
            best_recall = float(recall)
    return best_threshold, best_precision, best_recall, best_f1


def evaluate_predictions(y_true, y_score, threshold=0.5):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    precision, recall, f1 = precision_recall_f1_manual(y_true, y_pred)
    return {
        "auc_roc": roc_auc_manual(y_true, y_score),
        "auc_pr": average_precision_manual(y_true, y_score),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "threshold": threshold,
    }


def fit_logistic_irls(X, y, sample_weight=None, reg_strength=1.0, max_iter=50, tol=1e-7):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape
    Xb = np.column_stack([np.ones(n_samples), X])
    if sample_weight is None:
        sample_weight = np.ones(n_samples, dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        sample_weight = sample_weight / sample_weight.mean()
    weights = np.zeros(n_features + 1, dtype=float)
    eye = np.eye(n_features + 1)
    eye[0, 0] = 0.0
    for _ in range(max_iter):
        linear = Xb @ weights
        probabilities = sigmoid(linear)
        variance = np.clip(probabilities * (1.0 - probabilities), 1e-6, None)
        adjusted_weight = sample_weight * variance
        working_response = linear + (y - probabilities) / variance
        xtwx = Xb.T @ (Xb * adjusted_weight[:, None])
        xtwz = Xb.T @ (adjusted_weight * working_response)
        try:
            updated_weights = np.linalg.solve(xtwx + reg_strength * eye, xtwz)
        except np.linalg.LinAlgError:
            updated_weights = np.linalg.pinv(xtwx + reg_strength * eye) @ xtwz
        if np.max(np.abs(updated_weights - weights)) < tol:
            weights = updated_weights
            break
        weights = updated_weights
    return weights


def predict_proba(weights, X):
    X = np.asarray(X, dtype=float)
    Xb = np.column_stack([np.ones(X.shape[0]), X])
    return sigmoid(Xb @ weights)


def fit_and_score(train_df, val_df, feature_mode, reg_strength):
    X_train, feature_names, y_train, _, state = prepare_matrices(train_df, feature_mode)
    X_val, _, y_val, _, _ = prepare_matrices(val_df, feature_mode, state)
    positive_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    sample_weight = np.where(y_train == 1, positive_weight, 1.0)
    selected_feature_names = feature_names
    if feature_mode != "baseline":
        preliminary_weights = fit_logistic_irls(X_train, y_train, sample_weight=sample_weight, reg_strength=reg_strength)
        top_n = min(40, len(feature_names))
        selected_idx = np.argsort(np.abs(preliminary_weights[1:]))[::-1][:top_n]
        selected_idx = np.sort(selected_idx)
        selected_feature_names = [feature_names[index] for index in selected_idx]
        X_train = pd.DataFrame(X_train, columns=feature_names)[selected_feature_names].to_numpy(dtype=float)
        X_val = pd.DataFrame(X_val, columns=feature_names)[selected_feature_names].to_numpy(dtype=float)
    weights = fit_logistic_irls(X_train, y_train, sample_weight=sample_weight, reg_strength=reg_strength)
    val_prob = predict_proba(weights, X_val)
    threshold, precision, recall, f1 = best_threshold_for_f1(y_val, val_prob)
    metrics = evaluate_predictions(y_val, val_prob, threshold)
    return {
        "feature_mode": feature_mode,
        "reg_strength": reg_strength,
        "state": state,
        "weights": weights,
        "feature_names": selected_feature_names,
        "val_prob": val_prob,
        "threshold": threshold,
        "metrics": metrics,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def tune_model(train_df, val_df, feature_mode, reg_grid):
    results = []
    for reg_strength in reg_grid:
        result = fit_and_score(train_df, val_df, feature_mode, reg_strength)
        results.append(result)
        print(f"[{feature_mode}] reg={reg_strength:.3g} | AUC={result['metrics']['auc_roc']:.4f} | PR-AUC={result['metrics']['auc_pr']:.4f} | F1={result['metrics']['f1']:.4f}")
    results.sort(key=lambda item: (item["metrics"]["auc_roc"], item["metrics"]["f1"]), reverse=True)
    return results[0], pd.DataFrame([
        {"feature_mode": item["feature_mode"], "reg_strength": item["reg_strength"], **item["metrics"]}
        for item in results
    ])


def top_feature_importance(feature_names, weights, n=15):
    coefficients = pd.DataFrame({
        "feature": feature_names,
        "coefficient": weights[1:],
        "abs_coefficient": np.abs(weights[1:]),
    })
    return coefficients.sort_values("abs_coefficient", ascending=False).head(n).drop(columns=["abs_coefficient"])


def print_eda(train_df):
    missing_rate = train_df.isna().mean().sort_values(ascending=False)
    target_rate = train_df[TARGET_COL].mean()
    print("Dataset train shape:", train_df.shape)
    print("Dataset predict shape:", pd.read_csv(PREDICT_PATH).shape)
    print("Target positive rate:", round(float(target_rate), 4))
    print("Target distribution:")
    print(train_df[TARGET_COL].value_counts().to_string())
    print("\nTop missing values:")
    print(missing_rate.head(12).to_string())


def fit_full_model_and_predict(train_df, predict_df, feature_mode, reg_strength, threshold, output_path=OUTPUT_PATH):
    X_train, feature_names, y_train, _, state = prepare_matrices(train_df, feature_mode)
    positive_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    sample_weight = np.where(y_train == 1, positive_weight, 1.0)
    selected_feature_names = None
    if feature_mode != "baseline":
        preliminary_weights = fit_logistic_irls(X_train, y_train, sample_weight=sample_weight, reg_strength=reg_strength)
        top_n = min(40, len(feature_names))
        selected_idx = np.argsort(np.abs(preliminary_weights[1:]))[::-1][:top_n]
        selected_idx = np.sort(selected_idx)
        selected_feature_names = [feature_names[index] for index in selected_idx]
        X_train = pd.DataFrame(X_train, columns=feature_names)[selected_feature_names].to_numpy(dtype=float)
    else:
        selected_feature_names = feature_names
    weights = fit_logistic_irls(X_train, y_train, sample_weight=sample_weight, reg_strength=reg_strength)
    X_pred, _, _, groups_pred, _ = prepare_matrices(predict_df, feature_mode, state)
    if selected_feature_names is not None:
        X_pred = pd.DataFrame(X_pred, columns=feature_names)[selected_feature_names].to_numpy(dtype=float)
    pred_prob = predict_proba(weights, X_pred)
    pred_class = (pred_prob >= threshold).astype(int)
    output = pd.DataFrame({
        GROUP_COL: groups_pred,
        "predicted_probability": pred_prob,
        "predicted_class": pred_class,
    })
    output.to_csv(output_path, index=False)
    return output, weights, feature_names, state


def main(verbose=True):
    train_df = pd.read_csv(TRAIN_PATH)
    predict_df = pd.read_csv(PREDICT_PATH)
    if verbose:
        print_eda(train_df)
    train_split, val_split = stratified_group_split(train_df, test_size=0.2, random_state=42)
    if verbose:
        print("\nSplit by customer:")
        print(f"Train rows: {len(train_split)} | Val rows: {len(val_split)}")
        print(f"Train customers: {train_split[GROUP_COL].nunique()} | Val customers: {val_split[GROUP_COL].nunique()}")
        print(f"Train positive rate: {train_split[TARGET_COL].mean():.4f}")
        print(f"Val positive rate: {val_split[TARGET_COL].mean():.4f}")
    reg_grid = [0.1, 0.3, 1.0, 3.0]
    baseline_best, baseline_table = tune_model(train_split, val_split, "baseline", reg_grid)
    final_best, final_table = tune_model(train_split, val_split, "final", reg_grid)
    comparison = pd.DataFrame([
        {"model": "baseline", **baseline_best["metrics"], "reg_strength": baseline_best["reg_strength"]},
        {"model": "final", **final_best["metrics"], "reg_strength": final_best["reg_strength"]},
    ])
    if verbose:
        print("\nModel comparison:")
        print(comparison.to_string(index=False))
        print("\nBaseline grid search:")
        print(baseline_table.sort_values(["auc_roc", "f1"], ascending=False).to_string(index=False))
        print("\nFinal grid search:")
        print(final_table.sort_values(["auc_roc", "f1"], ascending=False).to_string(index=False))
        print("\nTop final feature coefficients:")
        importance = top_feature_importance(final_best["feature_names"], final_best["weights"], n=15)
        print(importance.to_string(index=False))
    else:
        importance = top_feature_importance(final_best["feature_names"], final_best["weights"], n=15)
    predictions, full_weights, full_feature_names, full_state = fit_full_model_and_predict(
        train_df=train_df,
        predict_df=predict_df,
        feature_mode="final",
        reg_strength=final_best["reg_strength"],
        threshold=final_best["threshold"],
        output_path=OUTPUT_PATH,
    )
    if verbose:
        print(f"\nPredictions saved to: {OUTPUT_PATH}")
        print(predictions.head().to_string(index=False))
    return {
        "comparison": comparison,
        "baseline_table": baseline_table,
        "final_table": final_table,
        "feature_importance": importance,
        "predictions": predictions,
        "output_path": OUTPUT_PATH,
    }


results = main(verbose=True)

Se imprimen los resultados para poder verlos

In [ ]:
print(results);

## Resumen de Resultados

La celda anterior imprime el resumen de EDA, la comparación baseline vs. final y la importancia de variables.

In [14]:
results["comparison"]

,model,auc_roc,auc_pr,precision,recall,f1,threshold,reg_strength
0,baseline,0.790848,0.484461,0.441420,0.519861,0.47744,0.635091,3.0
1,final,0.791711,0.486529,0.412988,0.567247,0.47798,0.598601,0.1


In [15]:
results["feature_importance"].head(15)

,feature,coefficient
15,income,0.950446
6,debt_to_income_ratio,0.527336
29,missing_income,0.446168
2,credit_score,0.442051
11,has_investment_account,0.383968
21,log1p_income,0.372922
13,has_personal_loan,-0.237420
19,log1p_credit_card_balance,-0.229963
0,avg_monthly_balance,0.225500
9,has_credit_card,0.170311


In [16]:
results["predictions"].head()

,customer_id,predicted_probability,predicted_class
0,602,0.351736,0
1,6001,0.267471,0
2,4975,0.342063,0
3,852,0.592574,0
4,3838,0.290308,0


## Conclusiones Finales

- El modelo final presenta una mejora marginal sobre la línea base en AUC-ROC y PR-AUC, utilizando una validación por cliente que reduce el riesgo de fuga de información entre meses de un mismo individuo.
- Las variables de mayor poder predictivo se asocian con ingreso, riesgo crediticio, balances, tenencia de productos y proxies de actividad o engagement, lo que sugiere que la propensión está vinculada tanto con capacidad financiera como con la profundidad de la relación bancaria.
- La ingeniería de variables agrega valor interpretativo y predictivo, aunque la mejora observada es moderada, lo cual indica que el dataset original ya concentra una señal relevante.
- El archivo de predicciones fue generado correctamente en `bank_product_propensity_predictions.csv`.

## Recomendaciones de Negocio

- Priorizar campañas sobre clientes con alta probabilidad estimada y perfiles consistentes en ingreso, saldo y engagement digital.
- Definir el umbral de clasificación en función del objetivo operativo: maximizar conversión, maximizar precisión o equilibrar ambos criterios según el presupuesto disponible.
- Analizar con mayor detalle los clientes con fuerte potencial predictivo pero con valores faltantes relevantes, ya que pueden constituir segmentos de interés comercial.
- Considerar, en desarrollos posteriores, modelos no lineales de mayor capacidad predictiva si el entorno de ejecución permite incorporar dependencias adicionales.

## Entregable

- Notebook final: `Parcial.ipynb`
- Predicciones finales: `bank_product_propensity_predictions.csv`